# Pause Detection Evaluation — FLEURS en_us (Comma-Only)

**Objective**: Evaluate how accurately our speech-prosody system's pause-detection
correlates with comma-implied pauses in the FLEURS dataset.

**Approach**:
1. Download FLEURS `en_us` test split directly from Hugging Face
2. Run each audio file through our **actual backend pipeline** (ASR → Pause Analyzer)
3. Sequence-align our system's word list against FLEURS' punctuated groundtruth
4. Score: Precision, Recall, F1 — comma positions only

**Pause Representation in Our System**:
- **Short pause** (gap < 0.5s): Whisper adds a comma to the word text (e.g. `"volume,"`)
- **Long pause** (gap ≥ 0.5s): Explicit `{"pause": "X.XXs"}` object inserted after the word

**Bug Fix Applied**: The worker normally runs PauseAnalyzer per-phrase, which silently
drops cross-phrase-boundary gaps. This notebook runs pause analysis on the **full flat
word list** in one pass.

## 1. Configuration

In [ ]:
# ── Configurable Parameters ──────────────────────────────────────────
N_SAMPLES = None            # Number of FLEURS test files to evaluate (set to None or 0 to run all ~647)
LONG_PAUSE_THRESHOLD = 0.5  # Seconds: gap >= this produces explicit {"pause":...} object
SAMPLE_RATE = 16000

# Google Drive paths (persisted across Colab disconnects)
DRIVE_BASE = "/content/drive/MyDrive/prosody_eval"
FLEURS_DIR = f"{DRIVE_BASE}/fleurs_en_us"
OUTPUT_DIR = f"{DRIVE_BASE}/outputs"
ERRORS_FILE = f"{DRIVE_BASE}/errors.json"
RESULTS_CSV = f"{DRIVE_BASE}/results.csv"

## 2. Environment Setup

In [ ]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')

# Create persistent directories
os.makedirs(FLEURS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'FLEURS dir: {FLEURS_DIR}')
print(f'Output dir: {OUTPUT_DIR}')

In [ ]:
# Install dependencies
!pip install -q faster-whisper torch pandas numpy matplotlib soundfile librosa huggingface_hub

In [ ]:
# Clone the prosody_interface repo (or pull latest if already cloned)
REPO_DIR = '/content/prosody_interface'
if os.path.isdir(REPO_DIR):
    print('Repo already cloned -- pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print('Cloning prosody_interface...')
    subprocess.run(
        ['git', 'clone', 'https://github.com/Abel-Jacob/prosody_interface.git', REPO_DIR],
        check=True
    )

# Add backend to Python path
BACKEND_DIR = os.path.join(REPO_DIR, 'backend')
if BACKEND_DIR not in sys.path:
    sys.path.insert(0, BACKEND_DIR)
print(f'Backend path: {BACKEND_DIR}')
print('Backend modules available for import.')

## 3. Load ASR Model (once)

In [ ]:
from pipeline.asr import load_asr_model, transcribe_chunk
from pipeline.prosody_pause import PauseAnalyzer

print('Loading ASR model (this may take a minute on first run)...')
asr_model = load_asr_model()  # Uses config defaults: medium.en on GPU, base.en on CPU
print('ASR model loaded.')

# Initialize PauseAnalyzer
pause_analyzer = PauseAnalyzer()
pause_analyzer.setup({})
print('PauseAnalyzer ready.')

## 4. Download FLEURS Dataset from Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download

TSV_LOCAL = os.path.join(FLEURS_DIR, 'test.tsv')
TAR_LOCAL = os.path.join(FLEURS_DIR, 'test.tar.gz')
AUDIO_EXTRACT_DIR = os.path.join(FLEURS_DIR, 'audio')

# Download test.tsv
if not os.path.exists(TSV_LOCAL):
    print('Downloading test.tsv from Hugging Face...')
    hf_hub_download(repo_id='google/fleurs', filename='data/en_us/test.tsv',
                    repo_type='dataset', local_dir=FLEURS_DIR)
    if not os.path.exists(TSV_LOCAL):
        actual = os.path.join(FLEURS_DIR, 'data', 'en_us', 'test.tsv')
        if os.path.exists(actual):
            import shutil; shutil.move(actual, TSV_LOCAL)
    print(f'test.tsv saved to {TSV_LOCAL}')
else:
    print(f'test.tsv already exists at {TSV_LOCAL}')

# Download test.tar.gz
if not os.path.exists(TAR_LOCAL):
    print('Downloading test.tar.gz from Hugging Face (this may take a few minutes)...')
    hf_hub_download(repo_id='google/fleurs', filename='data/en_us/audio/test.tar.gz',
                    repo_type='dataset', local_dir=FLEURS_DIR)
    if not os.path.exists(TAR_LOCAL):
        actual = os.path.join(FLEURS_DIR, 'data', 'en_us', 'audio', 'test.tar.gz')
        if os.path.exists(actual):
            import shutil; shutil.move(actual, TAR_LOCAL)
    print(f'test.tar.gz saved to {TAR_LOCAL}')
else:
    print(f'test.tar.gz already exists at {TAR_LOCAL}')

print(f'TSV size: {os.path.getsize(TSV_LOCAL):,} bytes')
print(f'TAR size: {os.path.getsize(TAR_LOCAL):,} bytes')

In [ ]:
# Extract audio files (using tar shell command -- much faster than Python tarfile)
os.makedirs(AUDIO_EXTRACT_DIR, exist_ok=True)
existing_wavs = [f for f in os.listdir(AUDIO_EXTRACT_DIR) if f.endswith('.wav')] if os.path.isdir(AUDIO_EXTRACT_DIR) else []

if len(existing_wavs) < 10:
    print('Extracting test.tar.gz...')
    !tar -xzf "{TAR_LOCAL}" -C "{AUDIO_EXTRACT_DIR}"
    existing_wavs = [f for f in os.listdir(AUDIO_EXTRACT_DIR) if f.endswith('.wav')]
    print(f'Extracted {len(existing_wavs)} audio files.')
else:
    print(f'Audio already extracted: {len(existing_wavs)} .wav files found.')

## 5. Parse FLEURS Metadata (test.tsv)

In [ ]:
import pandas as pd

FLEURS_COLUMNS = ['id', 'filename', 'raw_transcription', 'transcription', 'graphemes', 'num_samples', 'gender']
df_fleurs = pd.read_csv(TSV_LOCAL, sep='\t', header=None, names=FLEURS_COLUMNS)
print(f'FLEURS test set: {len(df_fleurs)} rows')
print('\nSample rows:')
df_fleurs[['id', 'filename', 'raw_transcription']].head(3)

In [ ]:
def find_audio_path(filename):
    """Locate audio file in the extracted directory."""
    direct = os.path.join(AUDIO_EXTRACT_DIR, filename)
    if os.path.exists(direct):
        return direct
    base = os.path.splitext(filename)[0]
    for ext in ['.wav', '.mp3', '.flac', '.ogg']:
        candidate = os.path.join(AUDIO_EXTRACT_DIR, base + ext)
        if os.path.exists(candidate):
            return candidate
    for root, dirs, files in os.walk(AUDIO_EXTRACT_DIR):
        for f in files:
            if f == filename or os.path.splitext(f)[0] == base:
                return os.path.join(root, f)
    return None

sample_file = df_fleurs.iloc[0]['filename']
sample_path = find_audio_path(sample_file)
print(f'Sample filename from TSV: {sample_file}')
print(f'Resolved audio path: {sample_path}')
if sample_path:
    print(f'File size: {os.path.getsize(sample_path):,} bytes')

## 6. Pipeline Function: `run_pipeline(audio_path)`

Calls the **actual backend** ASR + PauseAnalyzer functions.

**Bug fix applied**: Pause analysis runs on the **full flat word list** in one pass,
NOT per-phrase. This prevents silently dropping cross-phrase-boundary gaps.

In [ ]:
import numpy as np
import librosa
import json

def run_pipeline(audio_path):
    """
    Run the actual backend ASR + Pause pipeline on a single audio file.
    Returns the same JSON structure as the real interface output.
    """
    # Load audio (same as backend's worker._load_audio)
    audio, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

    # Stage 1: ASR transcription (uses our actual transcribe_chunk from pipeline.asr)
    asr_result = transcribe_chunk(audio, asr_model, language='en')
    asr_words = asr_result.get('words', [])
    full_text = asr_result.get('text', '')

    if not asr_words:
        return {'full_transcription': full_text, 'word_level_timestamps_and_stress': []}

    # Stage 2: Pause analysis on the FULL FLAT word list in one pass
    # BUG FIX: The worker normally runs PauseAnalyzer per-phrase, which drops
    # cross-phrase-boundary gaps. Here we run it on ALL words at once.
    pause_result = pause_analyzer.analyze(audio, asr_words)
    pause_words = pause_result.get('word_pauses', [])

    # Stage 3: Build output in the same format as our frontend's exported JSON
    word_list = []
    for i, asr_w in enumerate(asr_words):
        pause_after = 0.0
        is_hesitation = False
        if i < len(pause_words):
            pause_after = pause_words[i].get('pause_after', 0.0)
            is_hesitation = pause_words[i].get('is_hesitation', False)

        word_entry = {
            'word': asr_w['word'],
            'start_time': asr_w['start'],
            'end_time': asr_w['end'],
            'word_index': i,
            'pause_after': pause_after,
            'is_hesitation': is_hesitation,
            'asr_confidence': asr_w.get('confidence', 1.0),
        }
        word_list.append(word_entry)

        # Insert explicit pause object for long pauses (matches frontend behavior)
        if pause_after >= LONG_PAUSE_THRESHOLD:
            word_list.append({'pause': f'{pause_after:.2f}s'})

    return {
        'full_transcription': full_text,
        'word_level_timestamps_and_stress': word_list
    }

print('run_pipeline() defined -- uses actual backend ASR + PauseAnalyzer.')

## 7. Batch Process FLEURS Audio Files

In [ ]:
import time

eval_df = df_fleurs.copy() if (N_SAMPLES is None or N_SAMPLES <= 0) else df_fleurs.head(N_SAMPLES).copy()
print(f'Processing {len(eval_df)} files (N_SAMPLES={N_SAMPLES})')

errors_log = []
processed_count = 0
skipped_count = 0
failed_count = 0
t_batch_start = time.time()

for idx, row in eval_df.iterrows():
    filename = row['filename']
    file_id = os.path.splitext(filename)[0]
    output_path = os.path.join(OUTPUT_DIR, f'{file_id}.json')

    if os.path.exists(output_path):
        skipped_count += 1
        continue

    audio_path = find_audio_path(filename)
    if audio_path is None:
        errors_log.append({'filename': filename, 'error': 'Audio file not found'})
        failed_count += 1
        continue

    try:
        result = run_pipeline(audio_path)
        with open(output_path, 'w') as f:
            json.dump(result, f, indent=2)
        processed_count += 1
    except Exception as e:
        errors_log.append({'filename': filename, 'error': str(e)})
        failed_count += 1

    total_done = processed_count + skipped_count + failed_count
    if total_done % 10 == 0 or total_done == len(eval_df):
        elapsed = time.time() - t_batch_start
        print(f'  [{total_done}/{len(eval_df)}] processed={processed_count} '
              f'skipped={skipped_count} failed={failed_count} ({elapsed:.0f}s)')

if errors_log:
    with open(ERRORS_FILE, 'w') as f:
        json.dump(errors_log, f, indent=2)
    print(f'{len(errors_log)} errors logged to {ERRORS_FILE}')

elapsed_total = time.time() - t_batch_start
print(f'\nBatch complete: {processed_count} newly processed, '
      f'{skipped_count} skipped (cached), {failed_count} failed.')
rate = elapsed_total / max(1, processed_count)
print(f'Total time: {elapsed_total:.1f}s ({rate:.1f}s per file)')

## 8. Extraction Functions

### Groundtruth: Extract comma positions from FLEURS raw_transcription
### System: Extract all pause signals (short comma + long pause object)

In [ ]:
import re
import difflib

def _clean_word(w):
    """Strip punctuation for alignment comparison."""
    return re.sub(r"[^a-zA-Z0-9']", '', w).lower()

def extract_groundtruth_comma_points(raw_transcription, my_word_list):
    """
    Sequence-align FLEURS groundtruth words to our system's word list,
    return word_index positions (in OUR list) where FLEURS has a comma.
    Comma-only: ignores periods, question marks, etc.
    Normalizes spelling variations/homophones before alignment (Issue C).
    """
    gt_tokens = raw_transcription.split()
    gt_comma_positions = set()
    for i, token in enumerate(gt_tokens):
        stripped = token.rstrip('\'"')
        if stripped.endswith(','):
            gt_comma_positions.add(i)
    if not gt_comma_positions:
        return []
    gt_clean = [_clean_word(t) for t in gt_tokens]
    sys_clean = [_clean_word(w['word']) for w in my_word_list]
    
    # Normalize spelling variations/homophones before alignment (Issue C)
    sys_clean_normalized = list(sys_clean)
    for j, w_sys in enumerate(sys_clean):
        best_match = None
        best_ratio = 0.75
        start = max(0, j - 5)
        end = min(len(gt_clean), j + 6)
        for i in range(start, end):
            w_gt = gt_clean[i]
            ratio = difflib.SequenceMatcher(None, w_sys, w_gt).ratio()
            if ratio >= best_ratio:
                best_ratio = ratio
                best_match = w_gt
        if best_match is not None:
            sys_clean_normalized[j] = best_match

    matcher = difflib.SequenceMatcher(None, gt_clean, sys_clean_normalized)
    gt_to_sys = {}
    for block in matcher.get_matching_blocks():
        gt_start, sys_start, size = block
        for offset in range(size):
            gt_to_sys[gt_start + offset] = sys_start + offset
    comma_word_indices = []
    for gt_idx in sorted(gt_comma_positions):
        if gt_idx in gt_to_sys:
            comma_word_indices.append(gt_to_sys[gt_idx])
    return comma_word_indices

def extract_system_pause_signals(my_word_list_full):
    """
    Scan system's full word list (including pause objects) and return unified
    list of 'pause detected here' entries, excluding pauses on/after the last word
    and before the first word (trailing/leading silence).
    """
    signals = []
    word_entries_only = [e for e in my_word_list_full if 'word' in e]
    if not word_entries_only:
        return []
    last_word_idx = word_entries_only[-1]['word_index']
    explicit_pause_after = set()
    last_word_index = None
    for entry in my_word_list_full:
        if 'word' in entry:
            last_word_index = entry['word_index']
        elif 'pause' in entry and last_word_index is not None:
            if last_word_index >= last_word_idx:
                continue
            dur_str = entry['pause'].rstrip('s')
            try:
                duration = float(dur_str)
            except ValueError:
                duration = None
            signals.append({'after_word_index': last_word_index,
                            'detection_type': 'long_pause_object', 'duration': duration})
            explicit_pause_after.add(last_word_index)
    for entry in word_entries_only:
        word_text = entry['word'].strip()
        wi = entry['word_index']
        if wi >= last_word_idx:
            continue
        if word_text.endswith(',') and wi not in explicit_pause_after:
            signals.append({'after_word_index': wi,
                            'detection_type': 'short_pause_via_comma', 'duration': None})
    return signals

print('Extraction functions defined.')

## 9. Evaluation: Match, Score, Aggregate

In [ ]:
def evaluate_file(raw_transcription, system_output):
    """Evaluate pause detection for a single file."""
    wl = system_output.get('word_level_timestamps_and_stress', [])
    word_entries = [e for e in wl if 'word' in e]
    if not word_entries:
        return {'TP': 0, 'FP': 0, 'FN': 0, 'tp_long': 0, 'tp_short': 0, 'error': 'no words'}
    gt_comma_indices = set(extract_groundtruth_comma_points(raw_transcription, word_entries))
    sys_signals = extract_system_pause_signals(wl)
    sys_pause_indices = {s['after_word_index']: s for s in sys_signals}
    TP = FP = FN = tp_long = tp_short = 0
    for gt_idx in gt_comma_indices:
        if gt_idx in sys_pause_indices:
            TP += 1
            if sys_pause_indices[gt_idx]['detection_type'] == 'long_pause_object':
                tp_long += 1
            else:
                tp_short += 1
        else:
            FN += 1
    for sys_idx in sys_pause_indices:
        if sys_idx not in gt_comma_indices:
            FP += 1
    return {'TP': TP, 'FP': FP, 'FN': FN, 'tp_long': tp_long, 'tp_short': tp_short,
            'gt_count': len(gt_comma_indices), 'sys_count': len(sys_pause_indices)}

def compute_prf(tp, fp, fn):
    """Compute Precision, Recall, F1."""
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

print('Evaluation functions defined.')

In [ ]:
# Run evaluation across all processed files
file_results = []
total_tp_long = 0
total_tp_short = 0

for idx, row in eval_df.iterrows():
    filename = row['filename']
    raw_transcription = row['raw_transcription']
    file_id = os.path.splitext(filename)[0]
    output_path = os.path.join(OUTPUT_DIR, f'{file_id}.json')
    if not os.path.exists(output_path):
        continue
    with open(output_path, 'r') as f:
        system_output = json.load(f)
    result = evaluate_file(raw_transcription, system_output)
    if 'error' in result:
        continue
    p, r, f1 = compute_prf(result['TP'], result['FP'], result['FN'])
    file_results.append({
        'filename': filename, 'TP': result['TP'], 'FP': result['FP'], 'FN': result['FN'],
        'precision': round(p, 4), 'recall': round(r, 4), 'f1': round(f1, 4),
        'gt_commas': result['gt_count'], 'sys_signals': result['sys_count'],
        'tp_long': result['tp_long'], 'tp_short': result['tp_short'],
    })
    total_tp_long += result['tp_long']
    total_tp_short += result['tp_short']

print(f'Evaluated {len(file_results)} files successfully.')

## 10. Results Report

In [ ]:
import matplotlib.pyplot as plt

df_results = pd.DataFrame(file_results)

if len(df_results) == 0:
    print('ERROR: No files were evaluated. Check batch processing step.')
else:
    total_TP = int(df_results['TP'].sum())
    total_FP = int(df_results['FP'].sum())
    total_FN = int(df_results['FN'].sum())
    agg_p, agg_r, agg_f1 = compute_prf(total_TP, total_FP, total_FN)

    agg_row = pd.DataFrame([{
        'filename': 'OVERALL (micro-avg)', 'TP': total_TP, 'FP': total_FP, 'FN': total_FN,
        'precision': round(agg_p, 4), 'recall': round(agg_r, 4), 'f1': round(agg_f1, 4),
        'gt_commas': int(df_results['gt_commas'].sum()),
        'sys_signals': int(df_results['sys_signals'].sum()),
        'tp_long': total_tp_long, 'tp_short': total_tp_short,
    }])
    df_summary = pd.concat([df_results, agg_row], ignore_index=True)
    df_summary.to_csv(RESULTS_CSV, index=False)
    print(f'Results saved to {RESULTS_CSV}')

    print('\n' + '=' * 70)
    print('  COMMA-ONLY PAUSE DETECTION EVALUATION - FLEURS en_us test')
    print('=' * 70)
    print(f'\n  Files evaluated:  {len(df_results)}')
    print(f'  GT commas total:  {int(df_results.gt_commas.sum())}')
    print(f'  System signals:   {int(df_results.sys_signals.sum())}')
    print(f'\n  True Positives:   {total_TP}')
    print(f'  False Positives:  {total_FP}')
    print(f'  False Negatives:  {total_FN}')
    print(f'\n  Precision:  {agg_p:.4f}')
    print(f'  Recall:     {agg_r:.4f}')
    print(f'  F1 Score:   {agg_f1:.4f}')
    print('\n' + '-' * 70)
    print('  TP Breakdown by Detection Type:')
    print(f'    Long pause objects (gap >= {LONG_PAUSE_THRESHOLD}s): {total_tp_long}')
    print(f'    Short pause via comma (gap < {LONG_PAUSE_THRESHOLD}s):  {total_tp_short}')
    print('=' * 70)

In [ ]:
# Per-file summary table
if len(df_results) > 0:
    print('\n=== PER-FILE SUMMARY ===')
    display(df_summary.tail(min(20, len(df_summary))))

In [ ]:
# Visualization
if len(df_results) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # 1. P/R/F1 bar chart
    metrics = ['Precision', 'Recall', 'F1']
    values = [agg_p, agg_r, agg_f1]
    colors = ['#4F46E5', '#06B6D4', '#10B981']
    axes[0].bar(metrics, values, color=colors)
    axes[0].set_ylim(0, 1.05)
    axes[0].set_title('Aggregate Comma Pause Detection')
    for i, v in enumerate(values):
        axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')
    axes[0].grid(axis='y', linestyle='--', alpha=0.5)

    # 2. TP/FP/FN counts
    axes[1].bar(['TP', 'FP', 'FN'], [total_TP, total_FP, total_FN],
               color=['#10B981', '#EF4444', '#F59E0B'])
    axes[1].set_title('TP / FP / FN Counts')
    for i, v in enumerate([total_TP, total_FP, total_FN]):
        axes[1].text(i, v + 0.5, str(v), ha='center', fontweight='bold')
    axes[1].grid(axis='y', linestyle='--', alpha=0.5)

    # 3. TP breakdown by detection type
    tp_labels = [f'Long pause\n(>= {LONG_PAUSE_THRESHOLD}s)',
                 f'Short comma\n(< {LONG_PAUSE_THRESHOLD}s)']
    tp_vals = [total_tp_long, total_tp_short]
    axes[2].bar(tp_labels, tp_vals, color=['#6366F1', '#A78BFA'])
    axes[2].set_title('TP Breakdown by Detection Type')
    for i, v in enumerate(tp_vals):
        axes[2].text(i, v + 0.3, str(v), ha='center', fontweight='bold')
    axes[2].grid(axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

In [ ]:
# Statistical Sanity Checks
if len(df_results) > 0:
    print('\n=== SANITY CHECKS ===')
    warnings_found = False
    if total_TP == 0:
        print('[WARNING] Zero True Positives -- system may not be detecting any comma pauses.')
        warnings_found = True
    if agg_r < 0.1:
        print(f'[WARNING] Recall very low ({agg_r:.3f}) -- system is missing most GT commas.')
        print('          Possible causes: ASR not producing commas, or alignment issues.')
        warnings_found = True
    if agg_p < 0.1:
        print(f'[WARNING] Precision very low ({agg_p:.3f}) -- system produces many false pause signals.')
        warnings_found = True
    if total_tp_long == 0:
        print('[NOTE] Zero TPs from long pause objects -- all recall comes from short comma detection.')
        warnings_found = True
    if total_tp_short == 0:
        print('[NOTE] Zero TPs from short comma detection -- all recall comes from long pause objects.')
        warnings_found = True
    zero_gt = int((df_results['gt_commas'] == 0).sum())
    if zero_gt > 0:
        print(f'[NOTE] {zero_gt}/{len(df_results)} files had zero GT commas (only contribute FPs).')
        warnings_found = True
    if not warnings_found:
        print('All sanity checks passed -- no suspicious patterns detected.')
    print(f'\nMedian per-file F1: {df_results.f1.median():.4f}')
    print(f'Mean per-file F1:   {df_results.f1.mean():.4f}')
    print(f'Std per-file F1:    {df_results.f1.std():.4f}')

In [ ]:
# Per-file F1 distribution
if len(df_results) > 5:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(df_results['f1'], bins=20, color='#6366F1', edgecolor='white', alpha=0.85)
    ax.axvline(agg_f1, color='#EF4444', linestyle='--', linewidth=2,
               label=f'Aggregate F1 = {agg_f1:.3f}')
    ax.set_xlabel('F1 Score')
    ax.set_ylabel('Number of Files')
    ax.set_title('Per-File Comma Pause Detection F1 Distribution')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

## 11. Zero-F1 Diagnostic Breakdown

This diagnostic cell classifies files where $F_1 = 0$ into four distinct buckets to explain why the per-file $F_1$ distribution exhibits bimodal behavior:
1. **no_groundtruth_commas**: No commas to detect in the groundtruth raw transcription.
2. **no_system_signals**: The system output has zero pause signals of either type.
3. **alignment_mismatch**: Comma and system pause signals both exist but sequence-alignment failed to map them.
4. **genuine_miss**: Alignment succeeded, system signals and groundtruth commas both exist, but do not match.

In [ ]:
# ── Zero-F1 Diagnostic Breakdown ────────────────────────────────────
import os, json
import pandas as pd

zero_f1_files = df_results[df_results['f1'] == 0.0].copy()
total_zero_f1 = len(zero_f1_files)

buckets = {
    'no_groundtruth_commas': [],
    'no_system_signals': [],
    'alignment_mismatch': [],
    'genuine_miss': []
}

for idx, row in zero_f1_files.iterrows():
    filename = row['filename']
    file_id = os.path.splitext(filename)[0]
    output_path = os.path.join(OUTPUT_DIR, f'{file_id}.json')
    
    # Get Fleurs metadata row
    fleurs_row = df_fleurs[df_fleurs['filename'] == filename].iloc[0]
    raw_transcription = fleurs_row['raw_transcription']
    
    if not os.path.exists(output_path):
        continue
        
    with open(output_path, 'r') as f:
        system_output = json.load(f)
        
    wl = system_output.get('word_level_timestamps_and_stress', [])
    word_entries = [e for e in wl if 'word' in e]
    
    # 1. Count GT commas
    gt_tokens = raw_transcription.split()
    gt_comma_positions = []
    for i, token in enumerate(gt_tokens):
        stripped = token.rstrip('\'"')
        if stripped.endswith(','):
            gt_comma_positions.append(i)
            
    # 2. Count system signals
    sys_signals = extract_system_pause_signals(wl)
    
    # 3. Align and get mapped indices
    comma_word_indices = extract_groundtruth_comma_points(raw_transcription, word_entries)
    
    file_details = {
        'filename': filename,
        'raw_transcription': raw_transcription,
        'full_transcription': system_output.get('full_transcription', ''),
        'word_list': [{'word': e.get('word'), 'pause_after': e.get('pause_after'), 'pause': e.get('pause')} for e in wl],
        'gt_comma_positions': gt_comma_positions,
        'comma_word_indices': comma_word_indices,
        'sys_signals': sys_signals
    }
    
    if len(gt_comma_positions) == 0:
        buckets['no_groundtruth_commas'].append(file_details)
    elif len(sys_signals) == 0:
        buckets['no_system_signals'].append(file_details)
    elif len(comma_word_indices) == 0:
        buckets['alignment_mismatch'].append(file_details)
    else:
        buckets['genuine_miss'].append(file_details)

# Print overall count results
print('=' * 80)
print('  ZERO-F1 DIAGNOSTIC BUCKET COUNTS')
print('=' * 80)
for bname, flist in buckets.items():
    pct = (len(flist) / total_zero_f1 * 100) if total_zero_f1 > 0 else 0.0
    print(f'  {bname:<25} : {len(flist):3d} files ({pct:.1f}%)')
print('-' * 80)

# Summary Table
summary_data = []
for bname, flist in buckets.items():
    pct = (len(flist) / total_zero_f1 * 100) if total_zero_f1 > 0 else 0.0
    summary_data.append({
        'Bucket Name': bname,
        'File Count': len(flist),
        '% of Zero-F1 Files': f'{pct:.1f}%'
    })
df_diag_summary = pd.DataFrame(summary_data)
display(df_diag_summary)

# Detailed examples printout
for bname in ['no_system_signals', 'alignment_mismatch']:
    print('\n' + '=' * 80)
    print(f'  DETAILED DIAGNOSTICS: {bname.upper()} (Showing up to 5 examples)')
    print('=' * 80)
    examples = buckets[bname][:5]
    if not examples:
        print('  No files in this bucket.')
    for i, ex in enumerate(examples):
        print(f'\n[{i+1}] Filename: {ex["filename"]}')
        print(f'  FLEURS Groundtruth:   {ex["raw_transcription"]}')
        print(f'  System Transcription: {ex["full_transcription"]}')
        print(f'  GT Comma Positions:   {ex["gt_comma_positions"]}')
        print(f'  Mapped Word Indices:  {ex["comma_word_indices"]}')
        print(f'  System Pause Signals: {ex["sys_signals"]}')
        print(f'  System Word/Pause Entries:')
        for entry in ex['word_list'][:15]:
            if 'pause' in entry and entry['pause']:
                print(f'    - PAUSE: {entry["pause"]}')
            else:
                pause_after_str = f" (pause_after: {entry["pause_after"]}s)" if entry.get("pause_after") else ''
                print(f'    - Word: {entry["word"]}{pause_after_str}')
        if len(ex['word_list']) > 15:
            print('    ... (truncated)')
        print('-' * 50)


## 12. False Positive Spot-Check

This diagnostic cell draws a random sample of 8-10 false positive cases (where the system detected a pause but no groundtruth comma was nearby) to analyze why precision is low (e.g. plausible real pause vs. likely over-triggering).

In [ ]:
# ── False Positive Diagnostics ──────────────────────────────────────
import random

# Find files that have false positives (FP > 0)
fp_files = df_results[df_results['FP'] > 0].copy()

# Select up to 10 random files for spot checking
random.seed(42)  # Set seed for reproducible samples
sample_files = fp_files.sample(n=min(10, len(fp_files))).to_dict(orient='records') if len(fp_files) > 0 else []

print('=' * 80)
print(f'  FALSE POSITIVE SPOT-CHECK (Sample of {len(sample_files)} files)')
print('=' * 80)

for i, row in enumerate(sample_files):
    filename = row['filename']
    file_id = os.path.splitext(filename)[0]
    output_path = os.path.join(OUTPUT_DIR, f'{file_id}.json')
    
    with open(output_path, 'r') as f:
        system_output = json.load(f)
        
    wl = system_output.get('word_level_timestamps_and_stress', [])
    word_entries = [e for e in wl if 'word' in e]
    
    # Get Fleurs metadata row
    fleurs_row = df_fleurs[df_fleurs['filename'] == filename].iloc[0]
    raw_transcription = fleurs_row['raw_transcription']
    
    # Extract ground truth comma indices
    gt_comma_indices = set(extract_groundtruth_comma_points(raw_transcription, word_entries))
    
    # Extract system pause signals
    sys_signals = extract_system_pause_signals(wl)
    
    # Find which system signals are False Positives
    false_positives = [s for s in sys_signals if s['after_word_index'] not in gt_comma_indices]
    
    print(f'\n[{i+1}] Filename: {filename}')
    print(f'  FLEURS Groundtruth:   {raw_transcription}')
    print(f'  System Transcription: {system_output.get("full_transcription")}')
    print(f'  False Positive Detections ({len(false_positives)}):')
    
    for fp in false_positives:
        idx = fp['after_word_index']
        # Surrounding context (3-4 words)
        start_context = max(0, idx - 2)
        end_context = min(len(word_entries), idx + 3)
        context_words = []
        for j in range(start_context, end_context):
            w_str = word_entries[j]['word']
            if j == idx:
                w_str = f'**{w_str}**'
            context_words.append(w_str)
        context_str = ' '.join(context_words)
        
        det_type = fp['detection_type']
        if det_type == 'long_pause_object':
            det_type_str = f'long_pause_object ({fp["duration"]:.2f}s)'
        else:
            det_type_str = 'short_pause_via_comma'
            
        print(f'    - Word position: {idx} | Context: ... {context_str} ...')
        print(f'      Type: {det_type_str}')
        # Placeholder classifications (to be filled in manually during review)
    print('-' * 50)
